# Model-relative anomaly detection with `AnomalyDetector`

This notebook exercises the current `mnplib.anomalies.AnomalyDetector` API.

The detector uses **mismatch** as the anomaly criterion:

- for classification, an observation is anomalous when the predicted class differs from the observed class;
- for regression, an observation is anomalous when the observed and predicted values occupy different bins of one **common uniform discretization** of the observed target domain.

No anomaly-score threshold is used.

After detection, the notebook examines:

1. anomaly direction and common-bin diagnostics;
2. local correction information;
3. negative local explanatory gain;
4. miscoding-based explanation of the anomalous subset;
5. compressibility of the predicted states associated with anomalies;
6. classification anomalies;
7. the functional interface and model-based fitting.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.tree import DecisionTreeRegressor

from mnplib.anomalies import AnomalyDetector, anomaly_table

pd.set_option("display.max_columns", None)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 2. Helper functions

In [ ]:
def plot_regression_predictions(detector, title):
    """Plot observed and predicted values, highlighting detected anomalies."""
    anomalies = detector.anomalies()

    plt.figure(figsize=(11, 4))
    plt.plot(detector.y_, label="Observed", linewidth=1.5)
    plt.plot(detector.y_pred_, label="Predicted", linewidth=1.5)

    if anomalies.size:
        plt.scatter(
            anomalies,
            detector.y_[anomalies],
            marker="x",
            s=70,
            label="Anomalies",
        )

    plt.title(title)
    plt.xlabel("Sample index")
    plt.ylabel("Target")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_information_diagnostics(detector, title):
    """Plot information diagnostics for observations identified as anomalous."""
    table = detector.anomaly_table()

    if table.empty:
        print("No anomalies were detected.")
        return

    plt.figure(figsize=(11, 4))
    plt.scatter(
        table["sample_index"],
        table["local_correction_information"],
        label="Local correction information",
        s=45,
    )
    plt.scatter(
        table["sample_index"],
        table["negative_local_explanatory_gain"],
        label="Negative local explanatory gain",
        s=45,
    )

    plt.title(title)
    plt.xlabel("Sample index")
    plt.ylabel("Bits")
    plt.legend()
    plt.tight_layout()
    plt.show()


def display_explanation(explanation):
    """Display the main outputs returned by explain_anomalies()."""
    print("Status:", explanation["status"])
    print("Number of anomalies:", explanation["n_anomalies"])

    if explanation["status"] != "ok":
        return

    print("Selected features:", explanation["selected_features"])
    display(explanation["feature_analysis"])

    if not explanation["selection_path"].empty:
        display(explanation["selection_path"])

    display(explanation["subset_analysis"])

## 3. Regression: common-bin mismatch

The example below creates a billing-like regression problem. The supplied prediction vector represents the value expected from the main explanatory variables. Some observed values are then modified to create systematic differences between observed and predicted billing.

The detector does not use residual magnitude or a quantile threshold. It discretizes the observed target domain once and applies the same bin boundaries to both the observed and predicted values.

In [ ]:
n_samples = 180

usage = rng.uniform(0.0, 100.0, n_samples)
service_count = rng.integers(1, 6, n_samples)
financial_score = rng.normal(0.0, 1.0, n_samples)

discount_group = rng.choice(
    ["none", "moderate", "large"],
    size=n_samples,
    p=[0.72, 0.18, 0.10],
)

X_reg = pd.DataFrame(
    {
        "usage": usage,
        "service_count": service_count,
        "financial_score": financial_score,
        "discount_group": discount_group,
    }
)

expected_bill = (
    20.0
    + 0.75 * usage
    + 8.0 * service_count
    + rng.normal(0.0, 1.0, n_samples)
)

y_observed = expected_bill.copy()

moderate = discount_group == "moderate"
large = discount_group == "large"

y_observed[moderate] *= 0.75
y_observed[large] *= 0.50

# The prediction vector does not include the discount mechanism.
y_pred = expected_bill.copy()

print("R2 of supplied predictions:", r2_score(y_observed, y_pred))

In [ ]:
detector_reg = AnomalyDetector(
    task="regression",
    X_type="auto",
    y_type="numeric",
    n_bins="auto",
    random_state=RANDOM_SEED,
)

detector_reg.fit_predictions(X_reg, y_observed, y_pred)

detector_reg.summary()

### Common discretization

In [ ]:
print("Number of bins:", detector_reg.n_bins_)
print("Common bin edges:")
print(detector_reg.bin_edges_)

full_regression_table = detector_reg.anomaly_table(only_anomalies=False)

# Regression anomaly membership must be exactly equivalent to bin mismatch.
assert np.array_equal(
    full_regression_table["is_anomaly"].to_numpy(),
    (~full_regression_table["bin_match"]).to_numpy(),
)

full_regression_table.head(10)

In [ ]:
plot_regression_predictions(
    detector_reg,
    "Regression anomalies from common-bin mismatch",
)

## 4. Regression anomaly diagnostics

The table reports the mismatch decision together with two diagnostic quantities measured in bits:

- **local correction information**: the empirical information required to recover the observed state given the predicted state;
- **negative local explanatory gain**: the amount by which the prediction makes the observed state less expected than its marginal frequency would suggest.

Neither quantity participates in anomaly detection.

In [ ]:
regression_anomalies = detector_reg.anomaly_table()

regression_anomalies.sort_values(
    "local_correction_information",
    ascending=False,
).head(15)

In [ ]:
plot_information_diagnostics(
    detector_reg,
    "Information diagnostics for regression anomalies",
)

## 5. Under-predicted and over-predicted regression anomalies

In [ ]:
under_predicted = detector_reg.anomalies(kind="under_predicted")
over_predicted = detector_reg.anomalies(kind="over_predicted")

print("Under-predicted anomalies:", under_predicted.size)
print("Over-predicted anomalies:", over_predicted.size)

detector_reg.anomaly_table().groupby("direction").size()

## 6. Miscoding-based anomaly explanation

`explain_anomalies()` applies supervised miscoding only to the requested anomalous subset and ranks the available attributes according to the target used by the current `AnomalyDetector` implementation.

The result includes the individual feature analysis and the redundancy-aware subset selected by `Miscoding`.

In [ ]:
regression_explanation = detector_reg.explain_anomalies(
    max_features=3,
)

display_explanation(regression_explanation)

The explanation can also be computed separately for under-predicted and over-predicted anomalies.

In [ ]:
under_explanation = detector_reg.explain_anomalies(
    kind="under_predicted",
    max_features=3,
)

display_explanation(under_explanation)

## 7. Compressibility of anomalous predicted states

The public `anomaly_compressibility()` method compares two code lengths for the predicted states associated with anomalous observations:

\[
L_{\mathrm{optimal}}
=
-\sum_i \log_2 p(\hat y_i)
\]

and

\[
L_{\mathrm{uniform}}
=
n_A \log_2 |\mathcal{Y}|.
\]

The reported compression ratio is

\[
\frac{L_{\mathrm{optimal}}}{L_{\mathrm{uniform}}},
\]

and compressibility is its complement:

\[
1-\frac{L_{\mathrm{optimal}}}{L_{\mathrm{uniform}}}.
\]

A high compressibility indicates that model-relative anomalies tend to occur in a comparatively structured subset of predicted target states.

In [ ]:
regression_compressibility = detector_reg.anomaly_compressibility()
pd.Series(regression_compressibility)

## 8. Fitting from a supplied regression model

When `fit_model=True`, the estimator is cloned and fitted by the detector before predictions are generated.

In [ ]:
tree_detector = AnomalyDetector(
    task="regression",
    n_bins="auto",
    fit_model=True,
    random_state=RANDOM_SEED,
)

tree_detector.fit(
    X_reg[["usage", "service_count", "financial_score"]],
    y_observed,
    model=DecisionTreeRegressor(
        max_depth=4,
        random_state=RANDOM_SEED,
    ),
)

tree_detector.summary()

## 9. Classification: misclassification anomalies

In [ ]:
X_cls_array, y_cls = make_classification(
    n_samples=240,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
    class_sep=1.1,
    random_state=RANDOM_SEED,
)

X_cls = pd.DataFrame(
    X_cls_array,
    columns=[f"feature_{i}" for i in range(X_cls_array.shape[1])],
)

classifier = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_SEED,
)
classifier.fit(X_cls, y_cls)
y_cls_pred = classifier.predict(X_cls)

print("Training accuracy:", accuracy_score(y_cls, y_cls_pred))

In [ ]:
detector_cls = AnomalyDetector(
    task="classification",
    X_type="numeric",
    y_type="categorical",
    random_state=RANDOM_SEED,
)

detector_cls.fit_predictions(X_cls, y_cls, y_cls_pred)

detector_cls.summary()

In [ ]:
classification_table = detector_cls.anomaly_table()

classification_table.sort_values(
    "local_correction_information",
    ascending=False,
).head(15)

### Classification mismatch is the anomaly criterion

In [ ]:
full_classification_table = detector_cls.anomaly_table(only_anomalies=False)

assert np.array_equal(
    full_classification_table["is_anomaly"].to_numpy(),
    (~full_classification_table["correct"]).to_numpy(),
)

print("Misclassified observations:", detector_cls.anomalies().size)

### Classification anomaly explanation

In [ ]:
classification_explanation = detector_cls.explain_anomalies(
    max_features=3,
)

display_explanation(classification_explanation)

### Classification anomaly compressibility

In [ ]:
classification_compressibility = detector_cls.anomaly_compressibility()
pd.Series(classification_compressibility)

## 10. String class labels

In [ ]:
y_string = np.asarray([f"class_{value}" for value in y_cls])
y_string_pred = np.asarray([f"class_{value}" for value in y_cls_pred])

string_detector = AnomalyDetector(task="classification")
string_detector.fit_predictions(X_cls, y_string, y_string_pred)

string_detector.anomaly_table().head()

## 11. Functional interface

When predictions already exist, `anomaly_table()` provides a compact functional interface.

In [ ]:
quick_table = anomaly_table(
    X_reg,
    y_observed,
    y_pred,
    task="regression",
    n_bins="auto",
)

quick_table.head(10)

## 12. Optional automatic model selection

If the installed `mnplib` package provides the nescience-based automatic estimators, omitting both `model` and `predictions` delegates model construction to `NescienceRegressor` or `NescienceClassifier`.

In [ ]:
try:
    auto_detector = AnomalyDetector(
        task="regression",
        n_bins="auto",
        random_state=RANDOM_SEED,
    )
    auto_detector.fit(
        X_reg[["usage", "service_count", "financial_score"]],
        y_observed,
    )

    print("Automatic model type:", type(auto_detector.model_).__name__)
    display(pd.Series(auto_detector.summary()))
    display(auto_detector.anomaly_table().head())
except (ImportError, TypeError, ValueError) as exc:
    print("Automatic model-selection example skipped:")
    print(type(exc).__name__ + ":", exc)

## 13. Summary

The current workflow is intentionally separated into four operations:

```python
detector = AnomalyDetector(...)
detector.fit_predictions(X, y, predictions)

# Detection: mismatch only
detector.anomalies()
detector.anomaly_table()

# Information diagnostics
detector.summary()

# Attribute-based explanation
detector.explain_anomalies()

# Structure of anomalous predicted states
detector.anomaly_compressibility()
```

This keeps anomaly detection threshold-free while allowing the detected observations to be characterized and explained using information-theoretic diagnostics and supervised miscoding.